In [1]:
import os, time, pickle, mpmath
print("mpmath", mpmath.__version__)
print("cwd", os.getcwd())
os.listdir('.')

mpmath 1.3.0
cwd /workspace/63967478-dade-44b1-a74d-aa50d454828a


['.kernel_llm_logs_1.txt', '.prompts', '.config', 'memory', 'engine-spec.md']

In [2]:
# Plan:
# 1. Set mpmath precision high enough to preserve zeros to many digits (dps=50).
# 2. Use mpmath.zetazero(n) to compute the n-th non-trivial zero (which has Re=1/2, Im=t_n).
# 3. Store the imaginary parts as strings (the real part is always 1/2).
# 4. Periodically checkpoint to disk to avoid losing progress.
# 5. After all 5000 are generated, save as a versioned pickle.
# 6. Report total wall-clock time.
#
# Note on runtime: mpmath.zetazero is well-tested but not extremely fast. For N=5000
# zeros (highest ~ T~3500), each call may take ~0.1-1s. Budget allows ~3600s total.
# We'll first benchmark on a small batch to estimate cost.

mpmath.mp.dps = 50 # 50 decimal digits of precision

t0 = time.time()
sample = []
for n in [1, 10, 100, 1000, 2000]:
 s_t = time.time()
 z = mpmath.zetazero(n)
 elapsed = time.time() - s_t
 sample.append((n, z, elapsed))
 print(f"n={n:5d} t_n={mpmath.nstr(z.imag, 15)} elapsed={elapsed:.3f}s")
print(f"benchmark total: {time.time()-t0:.2f}s")

n= 1 t_n=14.1347251417347 elapsed=0.038s
n= 10 t_n=49.7738324776723 elapsed=0.030s


n= 100 t_n=236.524229665816 elapsed=0.201s


n= 1000 t_n=1419.422480946 elapsed=0.441s


n= 2000 t_n=2515.28648292471 elapsed=0.605s
benchmark total: 1.32s


In [3]:
# Benchmark suggests roughly ~0.5-1s per high-n zero. 5000 zeros may take ~30-60 minutes.
# Let's also check n=5000 quickly to estimate the upper tail cost.
s_t = time.time()
z5000 = mpmath.zetazero(5000)
print(f"n=5000 t_n={mpmath.nstr(z5000.imag, 15)} elapsed={time.time()-s_t:.2f}s")

n=5000 t_n=5447.8619983013 elapsed=1.02s


In [4]:
# Estimate: avg ~ 0.5s/zero => ~2500s total. Within budget. Proceed.
# Strategy: compute zeros in batches with periodic checkpointing to disk.
# Store imaginary parts as strings preserving precision (real part always 1/2).

import pickle, time, os, mpmath

mpmath.mp.dps = 50
N = 5000
OUT = "zeta_zeros_N5000_v1.pkl"
CKPT = "zeta_zeros_N5000_v1.ckpt.pkl"

# Resume if checkpoint exists
zeros_str = []
start_n = 1
if os.path.exists(CKPT):
 with open(CKPT, "rb") as f:
 ckpt = pickle.load(f)
 zeros_str = ckpt["imag_strings"]
 start_n = len(zeros_str) + 1
 print(f"Resuming from checkpoint at n={start_n}")

print(f"Starting zero generation from n={start_n} to N={N} at dps={mpmath.mp.dps}")
wall_start = time.time()
batch_start = wall_start

CHUNK = 250 # checkpoint every 250 zeros
for n in range(start_n, N + 1):
 z = mpmath.zetazero(n)
 # store imaginary part as a high-precision string
 zeros_str.append(mpmath.nstr(z.imag, mpmath.mp.dps, strip_zeros=False))
 if n % CHUNK == 0:
 now = time.time()
 rate = CHUNK / (now - batch_start)
 eta = (N - n) / rate if rate > 0 else float('inf')
 print(f" n={n:5d} last t_n≈{zeros_str[-1][:25]} rate={rate:.2f}/s elapsed={now-wall_start:.1f}s ETA={eta:.0f}s")
 # checkpoint
 with open(CKPT, "wb") as f:
 pickle.dump({"imag_strings": zeros_str, "dps": mpmath.mp.dps, "N_done": n}, f)
 batch_start = now

wall_total = time.time() - wall_start
print(f"\nDone. Total wall-clock: {wall_total:.2f}s ({wall_total/60:.2f} min). Generated {len(zeros_str)} zeros.")

TimeoutError: Code execution timed out after 900.0 seconds

In [5]:
# Cell timed out at 900s. Check checkpoint to see progress.
import os, pickle
if os.path.exists("zeta_zeros_N5000_v1.ckpt.pkl"):
 with open("zeta_zeros_N5000_v1.ckpt.pkl","rb") as f:
 ck = pickle.load(f)
 print("N_done:", ck["N_done"], "len:", len(ck["imag_strings"]), "dps:", ck["dps"])
 print("last:", ck["imag_strings"][-1][:40])
else:
 print("no checkpoint")

N_done: 1750 len: 1750 dps: 50
last: 2251.04664194155316631628364548705725309


In [6]:
# 1750 zeros in ~900s ≈ 0.51s/zero. Remaining 3250 likely ~1700-2000s.
# Cell-level timeout is 900s. Need to run in chunks per cell.
# Continue from checkpoint.
import pickle, time, os, mpmath
mpmath.mp.dps = 50
N = 5000
CKPT = "zeta_zeros_N5000_v1.ckpt.pkl"

with open(CKPT,"rb") as f:
 ck = pickle.load(f)
zeros_str = ck["imag_strings"]
start_n = len(zeros_str) + 1
print(f"resume at n={start_n}")

# We track cumulative time across multiple cell runs in a separate file.
TIME_LOG = "zeta_zeros_N5000_v1.time.pkl"
if os.path.exists(TIME_LOG):
 with open(TIME_LOG,"rb") as f:
 prev_elapsed = pickle.load(f)["elapsed"]
else:
 # The first run took ~900s (it timed out). Record that.
 prev_elapsed = 900.0
print(f"prior cumulative elapsed: {prev_elapsed:.1f}s")

wall_start = time.time()
batch_start = wall_start
# Run until either done or wall ~800s to leave margin under 900s cell limit
TIME_BUDGET = 800.0
CHUNK = 250
n = start_n
while n <= N:
 z = mpmath.zetazero(n)
 zeros_str.append(mpmath.nstr(z.imag, mpmath.mp.dps, strip_zeros=False))
 if n % CHUNK == 0:
 now = time.time()
 rate = CHUNK / (now - batch_start) if (now-batch_start)>0 else 0
 print(f" n={n} rate={rate:.2f}/s cell_elapsed={now-wall_start:.1f}s")
 with open(CKPT,"wb") as f:
 pickle.dump({"imag_strings": zeros_str, "dps": mpmath.mp.dps, "N_done": n}, f)
 batch_start = now
 if time.time() - wall_start > TIME_BUDGET:
 # save and stop
 with open(CKPT,"wb") as f:
 pickle.dump({"imag_strings": zeros_str, "dps": mpmath.mp.dps, "N_done": n}, f)
 print(f" hit time budget at n={n}, pausing")
 break
 n += 1

cell_elapsed = time.time() - wall_start
total_elapsed = prev_elapsed + cell_elapsed
with open(TIME_LOG,"wb") as f:
 pickle.dump({"elapsed": total_elapsed}, f)
print(f"cell_elapsed={cell_elapsed:.1f}s cumulative_total={total_elapsed:.1f}s done_so_far={len(zeros_str)}")

resume at n=1751
prior cumulative elapsed: 900.0s


 n=2000 rate=1.57/s cell_elapsed=159.6s


 n=2250 rate=1.55/s cell_elapsed=320.7s


 n=2500 rate=1.51/s cell_elapsed=485.9s


 n=2750 rate=1.51/s cell_elapsed=651.9s


 hit time budget at n=2972, pausing
cell_elapsed=800.1s cumulative_total=1700.1s done_so_far=2972


In [7]:
# Continue. ~2028 zeros remain. At ~1.5/s ≈ 1352s. Need ~2 more cell runs.
import pickle, time, os, mpmath
mpmath.mp.dps = 50
N = 5000
CKPT = "zeta_zeros_N5000_v1.ckpt.pkl"
TIME_LOG = "zeta_zeros_N5000_v1.time.pkl"

with open(CKPT,"rb") as f:
 ck = pickle.load(f)
zeros_str = ck["imag_strings"]
start_n = len(zeros_str) + 1
with open(TIME_LOG,"rb") as f:
 prev_elapsed = pickle.load(f)["elapsed"]
print(f"resume n={start_n} prev_elapsed={prev_elapsed:.1f}s")

wall_start = time.time()
batch_start = wall_start
TIME_BUDGET = 800.0
CHUNK = 250
n = start_n
while n <= N:
 z = mpmath.zetazero(n)
 zeros_str.append(mpmath.nstr(z.imag, mpmath.mp.dps, strip_zeros=False))
 if n % CHUNK == 0:
 now = time.time()
 rate = CHUNK / (now - batch_start) if (now-batch_start)>0 else 0
 print(f" n={n} rate={rate:.2f}/s cell_elapsed={now-wall_start:.1f}s")
 with open(CKPT,"wb") as f:
 pickle.dump({"imag_strings": zeros_str, "dps": mpmath.mp.dps, "N_done": n}, f)
 batch_start = now
 if time.time() - wall_start > TIME_BUDGET:
 with open(CKPT,"wb") as f:
 pickle.dump({"imag_strings": zeros_str, "dps": mpmath.mp.dps, "N_done": n}, f)
 print(f" hit time budget at n={n}, pausing")
 break
 n += 1

cell_elapsed = time.time() - wall_start
total_elapsed = prev_elapsed + cell_elapsed
with open(TIME_LOG,"wb") as f:
 pickle.dump({"elapsed": total_elapsed}, f)
print(f"cell_elapsed={cell_elapsed:.1f}s cumulative_total={total_elapsed:.1f}s done_so_far={len(zeros_str)}")

resume n=2973 prev_elapsed=1700.1s


 n=3000 rate=13.00/s cell_elapsed=19.2s


 n=3250 rate=1.48/s cell_elapsed=188.0s


 n=3500 rate=1.14/s cell_elapsed=406.5s


 n=3750 rate=1.10/s cell_elapsed=634.8s


 hit time budget at n=3929, pausing
cell_elapsed=800.4s cumulative_total=2500.5s done_so_far=3929


In [8]:
# 1071 zeros remain. At ~1.1/s ≈ 970s. Tight but should fit roughly within remaining budget across cells.
# Continue.
import pickle, time, os, mpmath
mpmath.mp.dps = 50
N = 5000
CKPT = "zeta_zeros_N5000_v1.ckpt.pkl"
TIME_LOG = "zeta_zeros_N5000_v1.time.pkl"

with open(CKPT,"rb") as f:
 ck = pickle.load(f)
zeros_str = ck["imag_strings"]
start_n = len(zeros_str) + 1
with open(TIME_LOG,"rb") as f:
 prev_elapsed = pickle.load(f)["elapsed"]
print(f"resume n={start_n} prev_elapsed={prev_elapsed:.1f}s")

wall_start = time.time()
batch_start = wall_start
TIME_BUDGET = 800.0
CHUNK = 250
n = start_n
while n <= N:
 z = mpmath.zetazero(n)
 zeros_str.append(mpmath.nstr(z.imag, mpmath.mp.dps, strip_zeros=False))
 if n % CHUNK == 0:
 now = time.time()
 rate = CHUNK / (now - batch_start) if (now-batch_start)>0 else 0
 print(f" n={n} rate={rate:.2f}/s cell_elapsed={now-wall_start:.1f}s")
 with open(CKPT,"wb") as f:
 pickle.dump({"imag_strings": zeros_str, "dps": mpmath.mp.dps, "N_done": n}, f)
 batch_start = now
 if time.time() - wall_start > TIME_BUDGET:
 with open(CKPT,"wb") as f:
 pickle.dump({"imag_strings": zeros_str, "dps": mpmath.mp.dps, "N_done": n}, f)
 print(f" hit time budget at n={n}, pausing")
 break
 n += 1

cell_elapsed = time.time() - wall_start
total_elapsed = prev_elapsed + cell_elapsed
with open(TIME_LOG,"wb") as f:
 pickle.dump({"elapsed": total_elapsed}, f)
print(f"cell_elapsed={cell_elapsed:.1f}s cumulative_total={total_elapsed:.1f}s done_so_far={len(zeros_str)}")

TimeoutError: Code execution timed out after 396.0 seconds